In [ ]:
from datetime import datetime
from opera_utils import disp
import geopandas as gpd
import os
from pathlib import Path
from transboundary_opera import displacement_tools as dt

%load_ext autoreload
%autoreload 2

In [ ]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")

In [ ]:
frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")

In [ ]:
# import asf_search as asf
# import re

# # Initialize a list to store frame IDs for each row
# all_frame_ids = []

# for entry, row in gdf.iterrows():
    
#     results = asf.geo_search(
#         intersectsWith=row.geometry.convex_hull.wkt,
#         dataset=asf.DATASET.OPERA_S1,
#         processingLevel=asf.PRODUCT_TYPE.DISP_S1,
#         maxResults=50  # We only need a few results to identify the Frame IDs
#     )
    
#     # Set for the current row
#     current_frame_ids = set()
#     pattern = re.compile(r'_F(\d{5})_')

#     for product in results:
#         filename = product.properties['fileName']
#         match = pattern.search(filename)
#         if match:
#             frame_id_str = match.group(1) 
#             current_frame_ids.add(int(frame_id_str))
    
#     # Add the sorted list of frame IDs to our collection
#     all_frame_ids.append(sorted(list(current_frame_ids)))

# # Assign the collected frame IDs to a new column in the GeoDataFrame
# gdf['frame_ids'] = all_frame_ids

In [ ]:
bbox = gdf.iloc[0].geometry.bounds

In [ ]:
results = disp.search(
        frame_id=frame_ids[0],
        start_datetime=datetime(2020, 1, 1),
        end_datetime=datetime(2025, 1, 1),
        url_type="s3",
        print_urls=False,
    )

In [ ]:
product_stack = disp.DispProductStack(results)

In [ ]:
import os

# Set credentials before running opera_utils
os.environ["EARTHDATA_USERNAME"] = ""
os.environ["EARTHDATA_PASSWORD"] = ''

In [ ]:
import subprocess

out_path = Path('/Users/clayc/Documents/Work/US_Mex_InSAR') / "OPERA_data"

download_dir = out_path / str(frame_ids[0])
os.makedirs(download_dir, exist_ok=True)

cmd = [
    "opera-utils", "disp-s1-download",
    "--output-dir", str(download_dir),
    "--frame-id", str(frame_ids[0]),
    "--start-datetime", "2020-01-01",
    "--end-datetime", "2025-01-01",
    # "--bbox", f"{round(bbox[0],2)}", f"{round(bbox[1],2)}", f"{round(bbox[2],2)}", f"{round(bbox[3],2)}",
    "--url-type", "HTTPS", # Change to "https" if you are not in AWS us-west-2
    "--num-workers", "8"
]

print("Starting download...")
subprocess.run(cmd, check=True)

In [ ]:
product_stack[0]

In [ ]:
product_stack.to_dataframe().head()